# AIMLC ZG5536 — Large Language Models for Generative AI
#### Assignment 2B — RAG Pipeline (Group 157)
**Domain:** Indian Financial Markets and Digital Payments (RBI / SEBI documents)

This notebook builds a small, complete RAG system on the Assignment 1A corpus: choose a chunk size, compare dense / BM25 / hybrid search, rerank, then generate grounded answers and check them with an LLM judge.

**Working method:** one stage at a time. Each stage is explained, run by the operator, and verified before the next one is added.

## Requirements map (20 marks)

| Part | What is built | Marks | Notebook stage |
|---|---|---|---|
| A | Dense search (FAISS) and chunk-size choice (100 vs 400 tokens) | 5 | Stages 1–3 |
| B | Dense vs BM25 vs hybrid (RRF) | 4 | Stage 4 |
| C | Cross-encoder reranking | 4 | Stage 5 |
| D | Grounded generation + LLM judge | 7 | Stages 6–7 |

**Reused from Assignment 1A (no new data):** the cleaned domain `.txt` files, 10 queries with expected answers from the 1A evaluation split, and the 1A fine-tuned model as the answer generator.

**Requirements authority:** `assignment-2b/concepts/Assignment 2B - RAG Pipeline.pdf`.
**Known PDF inconsistency:** the text says chunk sizes 100 and 400, one table says 100 and 500. This notebook uses **100 and 400** (the text) and discloses the discrepancy in Stage 3.

## How 1A connects to 2B

| Step | What changes | What it does **not** do |
|---|---|---|
| **CPT** (continued pretraining) | All weights keep learning next-token prediction on the finance corpus, so the model's word predictions fit the domain | It does not teach question/answer behaviour |
| **Instruction SFT** | Trains on question→answer pairs so the model answers briefly in the expected style | It does not give the model access to documents at answer time |
| **LoRA / QLoRA** | Trains small low-rank adapter matrices while the base weights stay frozen; QLoRA also stores the frozen base in 4-bit | It is a cheaper way to do SFT, not a different goal |
| **RAG** (this assignment) | At question time, retrieve relevant passages and put them in the prompt so the answer can be grounded in text | It retrains nothing |

A lower perplexity in 1A shows the model predicts domain text better. It does **not** show that the model answers questions correctly or stays faithful to supplied passages. Part D tests that directly.

---
## Stage 0 — Artifact check, environment and configuration

**Goal:** confirm that everything 2B reuses from 1A is present on this pod, record the compute resources, and fix one configuration that every later stage uses. No retrieval or model loading happens here.

| Cell | Purpose |
|---|---|
| 0A | Configuration: paths, seed, constants; locate the 1A repository |
| 0B | Resources and package versions (GPU, VRAM, host RAM, disk) |
| 0C | Artifact inventory: corpus, Q/A data, split files, CPT model, adapter |
| 0D | Load the corpus, save an input manifest with SHA-256 hashes |
| 0E | Model access and disk budget (no downloads yet) |
| 0F | Gate 0 summary |

In [ ]:
# Stage 0A: Configuration and path discovery
import os, sys, json, random, hashlib, time, platform
from pathlib import Path

SEED = 42                                   # same seed family as the 1A split (Stage 8E)
random.seed(SEED)

# 1A wrote everything to its persistent volume, mounted at /home/group157.
# Override with the ARTIFACT_ROOT environment variable if this pod differs.
ARTIFACT_ROOT = Path(os.environ.get("ARTIFACT_ROOT", "/home/group157"))
REPO_NAME     = "llm-assignment-1a-finance-corpus"

CORPUS_DIR    = ARTIFACT_ROOT / "domain_corpus"
CPT_DIR       = ARTIFACT_ROOT / "cpt_model"
ADAPTER_DIR   = ARTIFACT_ROOT / "qlora_finance_adapter"
QA_FILE       = ARTIFACT_ROOT / "instruction_dataset.jsonl"
WORK_DIR      = ARTIFACT_ROOT / "assignment_2b_outputs"     # all 2B artifacts go here
WORK_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_DOCS = 30
EXPECTED_QA   = {"total": 1196, "train": 957, "eval": 239}  # from saved 1A outputs (Stage 8E)

# Retrieval constants, fixed by the assignment PDF
CHUNK_SIZES   = [100, 400]       # tokens, no overlap (text of PDF; its table says 500, see intro)
TOP_K_CHUNK_EVAL = 3
TOP_K_COMPARE = 5
TOP_K_FUSE    = 20
RRF_K         = 60
TOP_K_RERANK  = 10
TOP_K_FINAL   = 3

print("Stage 0A: configuration")
print("-" * 40)
print(f"Python         : {platform.python_version()}")
print(f"Working dir    : {Path.cwd().resolve()}")
print(f"ARTIFACT_ROOT  : {ARTIFACT_ROOT.resolve() if ARTIFACT_ROOT.exists() else ARTIFACT_ROOT}  (exists={ARTIFACT_ROOT.exists()})")
print(f"2B output dir  : {WORK_DIR.resolve()}")

# Locate the 1A repository independently of ARTIFACT_ROOT
repo_hits = []
for base in {ARTIFACT_ROOT, Path.home(), Path.cwd()}:
    cand = base / REPO_NAME
    if cand.exists():
        repo_hits.append(cand.resolve())
REPO_DIR = repo_hits[0] if repo_hits else None
print(f"1A repository  : {REPO_DIR if REPO_DIR else 'NOT FOUND (searched ARTIFACT_ROOT, home, cwd)'}")

print("\nTop level of ARTIFACT_ROOT:")
if ARTIFACT_ROOT.exists():
    for p in sorted(ARTIFACT_ROOT.iterdir()):
        kind = "dir " if p.is_dir() else "file"
        print(f"  {kind}  {p.name}")
else:
    print("  (missing)")

STAGE0 = {}   # check name -> (status, detail), filled by later Stage 0 cells
print("\n>> [OK] Configuration set.")

In [ ]:
# Stage 0B: Resources and package versions
import shutil, importlib, subprocess

print("Stage 0B: compute resources")
print("-" * 40)

# --- GPU ---
gpu_info = {"available": False}
try:
    import torch
    gpu_info["torch"] = torch.__version__
    gpu_info["cuda_build"] = torch.version.cuda
    gpu_info["available"] = torch.cuda.is_available()
    if gpu_info["available"]:
        props = torch.cuda.get_device_properties(0)
        free_b, total_b = torch.cuda.mem_get_info(0)
        gpu_info.update(name=props.name, total_gb=round(total_b/1e9, 1), free_gb=round(free_b/1e9, 1),
                        bf16=torch.cuda.is_bf16_supported())
except Exception as e:
    gpu_info["error"] = repr(e)
print("GPU            :", gpu_info)

# --- Host RAM (physical and the container limit, which is what actually binds) ---
def read_first(paths):
    for p in paths:
        try:
            return Path(p).read_text().strip()
        except Exception:
            pass
    return None

mem_total_kb = None
try:
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemTotal"):
            mem_total_kb = int(line.split()[1])
        if line.startswith("MemAvailable"):
            mem_avail_kb = int(line.split()[1])
except Exception:
    mem_avail_kb = None
cg_limit = read_first(["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"])
cg_limit_gb = None
if cg_limit and cg_limit.isdigit() and int(cg_limit) < 1 << 60:
    cg_limit_gb = round(int(cg_limit)/1e9, 1)
print(f"Host RAM       : machine total {round(mem_total_kb/1e6,1) if mem_total_kb else '?'} GB | "
      f"available {round(mem_avail_kb/1e6,1) if mem_avail_kb else '?'} GB | container limit {cg_limit_gb or 'none/unknown'} GB")

# --- Persistent disk ---
disk_target = ARTIFACT_ROOT if ARTIFACT_ROOT.exists() else Path.cwd()
du = shutil.disk_usage(disk_target)
print(f"Persistent disk: {disk_target}  total {du.total/1e9:.1f} GB | free {du.free/1e9:.1f} GB")
hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
print(f"HF cache dir   : {hf_home}  (exists={hf_home.exists()})")

# --- Package versions ---
PKGS = ["numpy", "torch", "transformers", "peft", "bitsandbytes", "trl", "accelerate", "datasets",
        "sentence_transformers", "faiss", "rank_bm25", "huggingface_hub", "pandas"]
versions = {}
for name in PKGS:
    try:
        m = importlib.import_module(name)
        versions[name] = getattr(m, "__version__", "installed (no __version__)")
    except Exception:
        versions[name] = None
print("\nPackage versions:")
for k, v in versions.items():
    print(f"  {k:<22}{v if v else 'MISSING'}")

missing = [k for k, v in versions.items() if v is None]
print("\nMissing packages:", missing or "none")
print("Needed by 2B but not by 1A: sentence_transformers, faiss, rank_bm25 (install only those that are missing).")

(WORK_DIR / "environment_report.json").write_text(json.dumps(
    {"gpu": gpu_info, "mem_total_kb": mem_total_kb, "container_limit_gb": cg_limit_gb,
     "disk_free_gb": round(du.free/1e9, 1), "versions": versions, "python": platform.python_version()}, indent=2))
STAGE0["resources"] = ("PASS" if gpu_info.get("available") else "WARN", f"gpu={gpu_info.get('name','none')}")
print("\n>> [OK] Saved assignment_2b_outputs/environment_report.json")

In [ ]:
# Stage 0B-install (run only if the previous cell listed missing packages)
# Installs ONLY the missing 2B packages, and does not upgrade packages the 1A environment already pins.
INSTALL_MISSING = False                         # set True, run once, then re-run Stage 0B to confirm
TO_INSTALL = {"sentence_transformers": "sentence-transformers", "faiss": "faiss-cpu", "rank_bm25": "rank_bm25"}

if INSTALL_MISSING:
    for mod, pip_name in TO_INSTALL.items():
        if versions.get(mod) is None:
            print(f"Installing {pip_name} ...")
            r = subprocess.run([sys.executable, "-m", "pip", "install", pip_name],
                               capture_output=True, text=True)
            print(r.stdout[-600:] if r.returncode == 0 else r.stderr[-1200:])
    print("\nRestart the kernel if torch/transformers/numpy versions changed, then re-run from Stage 0A.")
else:
    print("INSTALL_MISSING is False: nothing installed.")

In [ ]:
# Stage 0C: Inventory of 1A artifacts (corpus, Q/A data, split, CPT model, adapter)
def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def human(n):
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024

def jsonl_rows(p):
    rows = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

print("Stage 0C: artifact inventory")
print("-" * 40)

# --- Corpus ---
corpus_files = sorted(CORPUS_DIR.glob("*.txt")) if CORPUS_DIR.exists() else []
print(f"Corpus dir     : {CORPUS_DIR}  files={len(corpus_files)} (expected {EXPECTED_DOCS})")
STAGE0["corpus"] = ("PASS" if len(corpus_files) == EXPECTED_DOCS else "FAIL", f"{len(corpus_files)} files")

# --- Q/A file(s) ---
qa_rows = jsonl_rows(QA_FILE) if QA_FILE.exists() else []
print(f"\nQ/A file       : {QA_FILE.name}  rows={len(qa_rows)} (expected {EXPECTED_QA['total']})")
if qa_rows:
    print("  keys of row 0:", sorted(qa_rows[0].keys()))
    print("  note: the submitted 1A file holds only instruction/response; source_document and evidence are in the repository's curated file.")
STAGE0["qa_full"] = ("PASS" if len(qa_rows) == EXPECTED_QA["total"] else "FAIL", f"{len(qa_rows)} rows")

print("\nOther instruction*.jsonl files under ARTIFACT_ROOT (split exports, if 1A wrote them):")
split_candidates = sorted(ARTIFACT_ROOT.glob("instruction_dataset*.jsonl")) if ARTIFACT_ROOT.exists() else []
for p in split_candidates:
    print(f"  {p.name:<45} rows={len(jsonl_rows(p))}")
if not split_candidates:
    print("  none")

curated = None
if REPO_DIR:
    c = REPO_DIR / "data" / "instruction" / "instruction_dataset_curated.jsonl"
    curated = c if c.exists() else None
print(f"\nCurated metadata file (with evidence): {curated if curated else 'NOT FOUND'}")
if curated:
    cur_rows = jsonl_rows(curated)
    same_order = (len(cur_rows) == len(qa_rows) and
                  all(a["instruction"] == b["instruction"] and a["response"] == b["response"]
                      for a, b in zip(cur_rows, qa_rows)))
    print(f"  rows={len(cur_rows)} | ordered Q/A identical to {QA_FILE.name}: {same_order}")
    STAGE0["curated_matches_qa"] = ("PASS" if same_order else "FAIL", f"rows={len(cur_rows)}")
else:
    STAGE0["curated_matches_qa"] = ("PENDING", "curated metadata not located")

# --- CPT model ---
print(f"\nCPT model dir  : {CPT_DIR}")
if CPT_DIR.exists():
    total = 0
    for p in sorted(CPT_DIR.iterdir()):
        total += p.stat().st_size
        print(f"  {p.name:<35}{human(p.stat().st_size)}")
    print(f"  total {human(total)}")
    cfg_p = CPT_DIR / "config.json"
    if cfg_p.exists():
        cfg = json.loads(cfg_p.read_text())
        print("  config:", {k: cfg.get(k) for k in ["architectures", "vocab_size", "max_position_embeddings",
                                                    "hidden_size", "num_hidden_layers", "pad_token_id", "eos_token_id"]})
    STAGE0["cpt_model"] = ("PASS" if (CPT_DIR / "config.json").exists() else "FAIL", f"{human(total)}")
else:
    STAGE0["cpt_model"] = ("FAIL", "directory missing")

# --- Adapter ---
print(f"\nAdapter dir    : {ADAPTER_DIR}")
if ADAPTER_DIR.exists():
    for p in sorted(ADAPTER_DIR.iterdir()):
        print(f"  {p.name:<35}{human(p.stat().st_size)}")
    ac = ADAPTER_DIR / "adapter_config.json"
    if ac.exists():
        a = json.loads(ac.read_text())
        print("  adapter_config:", {k: a.get(k) for k in ["base_model_name_or_path", "r", "lora_alpha",
                                                          "target_modules", "lora_dropout"]})
    STAGE0["adapter"] = ("PASS" if ac.exists() and (ADAPTER_DIR / "adapter_model.safetensors").exists() else "FAIL", "")
else:
    STAGE0["adapter"] = ("FAIL", "directory missing")

# --- Tokenizer saved with the adapter ---
tok_cfg = ADAPTER_DIR / "tokenizer_config.json"
print(f"\nTokenizer saved with adapter: {tok_cfg.exists()}")
print("\n>> [OK] Inventory printed. Any FAIL above must be resolved before Gate 0.")

In [ ]:
# Stage 0D: Load the corpus and save an input manifest
docs = {}
manifest = []
for p in corpus_files:
    text = p.read_text(encoding="utf-8")
    docs[p.name] = text
    manifest.append({"doc_id": p.name, "chars": len(text), "bytes": p.stat().st_size, "sha256": sha256_file(p)})

assert len(docs) == EXPECTED_DOCS, f"expected {EXPECTED_DOCS} documents, found {len(docs)}"

total_chars = sum(m["chars"] for m in manifest)
print("Stage 0D: corpus loaded")
print("-" * 40)
print(f"Documents      : {len(docs)}")
print(f"Total characters: {total_chars:,}")
print("\nFive largest / five smallest documents (characters):")
by_size = sorted(manifest, key=lambda m: m["chars"])
for m in by_size[:5] + by_size[-5:]:
    print(f"  {m['chars']:>10,}  {m['doc_id'][:90]}")

provenance = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "artifact_root": str(ARTIFACT_ROOT), "seed": SEED, "documents": manifest,
    "qa_file_sha256": sha256_file(QA_FILE) if QA_FILE.exists() else None,
}
(WORK_DIR / "input_manifest.json").write_text(json.dumps(provenance, indent=2))
STAGE0["docs_loaded"] = ("PASS", f"{len(docs)} docs, {total_chars:,} chars")
print("\n>> [OK] Saved assignment_2b_outputs/input_manifest.json")

In [ ]:
# Stage 0E: Model access and disk budget (no weights are downloaded here)
PLANNED_MODELS = [
    # (role, model id, stage, intended precision)
    ("embedder",            "BAAI/bge-small-en-v1.5",                  "Stage 2",  "fp32/fp16 on GPU"),
    ("cross-encoder",       "cross-encoder/ms-marco-MiniLM-L-6-v2",    "Stage 5",  "fp32/fp16 on GPU"),
    ("cross-encoder (alt)", "cross-encoder/ms-marco-MiniLM-L6-v2",     "Stage 5",  "fp32/fp16 on GPU"),
    ("generator fallback",  "Qwen/Qwen2.5-7B-Instruct",                "Stage 6",  "bf16 or 4-bit"),
    ("judge candidate",     "meta-llama/Llama-3.1-8B-Instruct",        "Stage 7",  "bf16 or 4-bit"),
]
print("Stage 0E: model access and download size (metadata only)")
print("-" * 60)
rows = []
try:
    from huggingface_hub import HfApi
    api = HfApi()
    for role, mid, stage, prec in PLANNED_MODELS:
        try:
            info = api.model_info(mid, files_metadata=True)
            weights = [s for s in info.siblings if s.rfilename.endswith((".safetensors", ".bin")) and s.size]
            size_gb = sum(s.size for s in weights) / 1e9
            gated = bool(getattr(info, "gated", False))
            rows.append((role, mid, stage, prec, "reachable", f"{size_gb:.2f} GB", "gated" if gated else "open"))
        except Exception as e:
            rows.append((role, mid, stage, prec, "NOT reachable", "?", type(e).__name__))
except Exception as e:
    print("huggingface_hub unavailable or network blocked:", repr(e))

for r in rows:
    print(f"{r[0]:<20}{r[1]:<40}{r[2]:<8}{r[3]:<18}{r[4]:<15}{r[5]:<10}{r[6]}")

free_gb = shutil.disk_usage(hf_home if hf_home.exists() else Path.home()).free / 1e9
print(f"\nFree disk where the HF cache lives: {free_gb:.1f} GB")
print("Reminder: a 4-bit runtime load still downloads the full-precision checkpoint unless a pre-quantized repo is used.")
print("Gated models (e.g. Llama) need an accepted licence and an HF token. Resolve this before Stage 7.")

ok = bool(rows) and any(r[4] == "reachable" for r in rows)
STAGE0["model_access"] = ("PASS" if ok else "WARN", f"{sum(r[4]=='reachable' for r in rows)}/{len(rows)} reachable")
print("\n>> [OK] Access check complete. Nothing downloaded.")

In [ ]:
# Stage 0F: Gate 0 summary
print("Stage 0 gate summary")
print("-" * 60)
for k, (status, detail) in STAGE0.items():
    print(f"{status:<8}{k:<24}{detail}")

hard_fail = [k for k, (s, _) in STAGE0.items() if s == "FAIL"]
print()
if hard_fail:
    print("Gate 0 NOT passed. Resolve:", hard_fail)
else:
    print("No hard failures. Review any WARN / PENDING rows with the reviewer before starting Stage 1.")

(WORK_DIR / "stage0_summary.json").write_text(json.dumps(STAGE0, indent=2))